# Test Proxies

# Load Proxies

In [ ]:
import csv

import csv

def load_proxies(file):

    proxies = []

    with open(file, "r", encoding="utf-8") as f:

        reader = csv.reader(f)

        for row in reader:

            if not row:  # bỏ dòng trống
                continue

            proxy = row[0].strip()

            if proxy:
                proxies.append(proxy)

    return proxies


proxy_list = load_proxies("proxies.csv")

print("Total proxies:", len(proxy_list))
print(proxy_list[:5])  # test 5 proxy đầu

# Hàm Test Proxies

In [ ]:
import requests

def test_proxy(proxy):

    proxies = {
        "http": proxy,
        "https": proxy
    }

    try:

        r = requests.get(
            "https://api.ipify.org",
            proxies=proxies,
            timeout=5
        )

        if r.status_code == 200:

            print("WORKING:", proxy, "IP:", r.text)

            return True

    except:
        pass

    print("DEAD:", proxy)

    return False

# Test và lọc proxies

In [ ]:
working = []

for proxy in proxy_list:

    if test_proxy(proxy):

        working.append(proxy)

print("Working proxies:", len(working))

# Crawl Data

In [ ]:
import requests
import csv
import random
import time
import threading
from fake_useragent import UserAgent

ua = UserAgent()

OUTPUT_FILE = "lazada_products_dataset_12.csv"
KEYWORD_PER_PROXY = 5
THREADS = 4
MAX_PAGE = 15
MAX_RETRY = 5


# ----------------------------
# LOAD FILES
# ----------------------------

def load_list(file):

    data = []

    with open(file, encoding="utf8") as f:

        for line in f:

            line = line.strip()

            if line:
                data.append(line)

    return data


proxies = load_list("working_proxies.txt")
keywords = load_list("keyword_1.txt")

print("Proxies:", len(proxies))
print("Keywords:", len(keywords))


# ----------------------------
# CSV
# ----------------------------

lock = threading.Lock()

def init_csv():

    with open(OUTPUT_FILE, "w", newline="", encoding="utf8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "keyword",
            "name",
            "price",
            "rating",
            "sold",
            "url"
        ])


def save_csv(rows):

    with lock:

        with open(OUTPUT_FILE, "a", newline="", encoding="utf8") as f:

            writer = csv.writer(f)

            writer.writerows(rows)


# ----------------------------
# PARSER
# ----------------------------

def parse_products(data, keyword):

    rows = []

    try:

        items = data.get("mods", {}).get("listItems", [])

        for p in items:

            rows.append([
                keyword,
                p.get("name"),
                p.get("price"),
                p.get("ratingScore"),
                p.get("itemSoldCntShow"),
                "https:" + (p.get("itemUrl") or p.get("productUrl") or "")
            ])

    except:
        pass

    return rows


# ----------------------------
# GET RANDOM PROXY
# ----------------------------

def get_proxy():

    proxy = random.choice(proxies)

    return {
        "http": proxy,
        "https": proxy
    }


# ----------------------------
# CRAWL
# ----------------------------

def crawl(keyword):

    for page in range(1, MAX_PAGE + 1):

        url = f"https://www.lazada.vn/catalog/?ajax=true&q={keyword}&page={page}"

        success = False

        for attempt in range(MAX_RETRY):

            headers = {
                "User-Agent": ua.random
            }

            proxy = get_proxy()

            try:

                r = requests.get(
                    url,
                    headers=headers,
                    proxies=proxy,
                    timeout=20
                )

                if r.status_code != 200:
                    raise Exception("Bad status")

                data = r.json()

                rows = parse_products(data, keyword)

                if rows:
                    save_csv(rows)

                print(f"OK: {keyword} | page {page}")

                success = True
                break

            except Exception as e:

                print(f"Retry {attempt+1} | {keyword} | page {page}")

                time.sleep(random.uniform(4,8))

        if not success:
            print(f"FAIL: {keyword} | page {page}")

        # delay để tránh block
        time.sleep(random.uniform(6,12))


# ----------------------------
# WORKER
# ----------------------------

def worker(keyword_batch):

    for k in keyword_batch:

        crawl(k)

        # delay giữa keyword
        time.sleep(random.uniform(30,70))


# ----------------------------
# SPLIT KEYWORDS
# ----------------------------

def split_keywords():

    batches = []

    for i in range(0, len(keywords), KEYWORD_PER_PROXY):

        batches.append(keywords[i:i + KEYWORD_PER_PROXY])

    return batches


# ----------------------------
# MAIN
# ----------------------------

def main():

    init_csv()

    batches = split_keywords()

    threads = []

    for batch in batches:

        t = threading.Thread(
            target=worker,
            args=(batch,)
        )

        t.start()

        threads.append(t)

        if len(threads) >= THREADS:

            for t in threads:
                t.join()

            threads = []

    for t in threads:
        t.join()

    print("DONE")


if __name__ == "__main__":
    main()

Proxies: 162
Keywords: 35
Retry 1 | chân váy chữ A | page 1Retry 1 | áo khoác bomber | page 1

Retry 2 | áo khoác bomber | page 1
Retry 2 | chân váy chữ A | page 1
Retry 3 | chân váy chữ A | page 1
Retry 1 | son môi Hàn Quốc | page 1
Retry 1 | ví nữ cầm tay | page 1
Retry 4 | chân váy chữ A | page 1
Retry 2 | son môi Hàn Quốc | page 1
Retry 2 | ví nữ cầm tay | page 1
Retry 3 | áo khoác bomber | page 1
Retry 3 | son môi Hàn Quốc | page 1
Retry 4 | áo khoác bomber | page 1
Retry 4 | son môi Hàn Quốc | page 1
Retry 5 | áo khoác bomber | page 1
Retry 5 | chân váy chữ A | page 1
FAIL: chân váy chữ A | page 1
FAIL: áo khoác bomber | page 1
Retry 3 | ví nữ cầm tay | page 1
Retry 4 | ví nữ cầm tay | page 1
Retry 1 | áo khoác bomber | page 2
Retry 5 | son môi Hàn Quốc | page 1
FAIL: son môi Hàn Quốc | page 1
Retry 1 | chân váy chữ A | page 2
Retry 2 | áo khoác bomber | page 2
Retry 3 | áo khoác bomber | page 2
OK: son môi Hàn Quốc | page 2
Retry 4 | áo khoác bomber | page 2
Retry 5 | ví nữ cầm 